# 📈 Stock Price Visualization & Future Prediction

This notebook demonstrates:
1. **Stock Price Visualization** - Historical data with interactive charts
2. **Model Training** - Train Transformer and LSTM models
3. **Future Predictions** - Generate and visualize predictions
4. **Performance Analysis** - Compare model predictions

In [5]:
# Cell 1: Environment Setup and Imports
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
warnings.filterwarnings('ignore')

# Set up directory
current_dir = os.getcwd()
if 'Trading_Project' in current_dir and not current_dir.endswith('QuantStock'):
    quantstock_dir = os.path.join(current_dir, 'QuantStock')
    if os.path.exists(quantstock_dir):
        os.chdir(quantstock_dir)
        current_dir = os.getcwd()

if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Working Directory: {current_dir}")

# Install required packages if not available
try:
    import torch
    import plotly
    import yfinance as yf
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'plotly', 'yfinance', 'scikit-learn', 'seaborn'])
    import torch
    import plotly
    import yfinance as yf

# Import project modules
from utils.tools import dict_to_namespace
from stock_data_handle import Stock_Data
from pm.PM_transformer import PM_Transformer
from pm.PM_lstm import PM_LSTM

print("✅ All imports successful!")
print(f"📊 PyTorch: {torch.__version__}")
print(f"📈 Plotly: {plotly.__version__}")

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: numpy._core.multiarray failed to import

## 📊 Step 1: Load and Visualize Stock Data

In [ ]:
# Cell 2: Load Stock Data
print("🔄 Loading stock data...")

# Load configuration
with open("model_config/transformer_config.json", 'r') as f:
    config = json.load(f)
args = dict_to_namespace(config)

# Load stock data
project_name = args.project_name
stock_data = Stock_Data(
    dataset_name=args.data_dict[project_name]["dataset_name"],
    full_stock_path=args.data_dict[project_name]["full_stock_path"],
    window_size=args.seq_len,
    root_path=args.root_path,
    prediction_len=args.prediction_len,
    scale=True
)

print(f"✅ Stock data loaded successfully!")
print(f"📊 Data shape: {stock_data.data.shape}")
print(f"📈 Number of stocks: {stock_data.data.shape[1]}")
print(f"📅 Time steps: {stock_data.data.shape[0]}")
print(f"📋 Features per stock: {stock_data.data.shape[2]}")

🔄 Loading stock data...


NameError: name 'dict_to_namespace' is not defined

In [ ]:
# Cell 3: Create Interactive Stock Price Visualization
print("📊 Creating interactive stock price visualizations...")

# Get stock symbols
stock_symbols = [f'STOCK_{i+1}' for i in range(min(5, stock_data.data.shape[1]))]

# Create sample price data from the loaded data
price_data = stock_data.data[:, :len(stock_symbols), 0]  # First feature for visualization

# Create date range
dates = pd.date_range(start='2020-01-01', periods=len(price_data), freq='D')

# Create interactive plot
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Stock Prices', 'Volume', 'Returns', 'Moving Averages')
)

# Plot 1: Stock Prices
for i, symbol in enumerate(stock_symbols):
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=price_data[:, i],
            mode='lines',
            name=symbol,
            line=dict(width=2)
        ),
        row=1, col=1
    )

# Plot 2: Volume (if available)
if stock_data.data.shape[2] > 1:
    volume_data = stock_data.data[:, :len(stock_symbols), 1]
    for i, symbol in enumerate(stock_symbols):
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=volume_data[:, i],
                mode='lines',
                name=f'{symbol} Volume',
                line=dict(width=1),
                showlegend=False
            ),
            row=1, col=2
        )

# Plot 3: Returns
returns_data = np.diff(price_data, axis=0) / price_data[:-1] * 100
for i, symbol in enumerate(stock_symbols):
    fig.add_trace(
        go.Scatter(
            x=dates[1:],
            y=returns_data[:, i],
            mode='lines',
            name=f'{symbol} Returns',
            line=dict(width=1),
            showlegend=False
        ),
        row=2, col=1
    )

# Plot 4: Moving Averages
for i, symbol in enumerate(stock_symbols):
    ma_short = pd.Series(price_data[:, i]).rolling(window=20).mean()
    ma_long = pd.Series(price_data[:, i]).rolling(window=50).mean()
    
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=ma_short,
            mode='lines',
            name=f'{symbol} MA20',
            line=dict(width=1, dash='dash'),
            showlegend=False
        ),
        row=2, col=2
    )

fig.update_layout(
    title='📈 Stock Market Analysis Dashboard',
    height=800,
    showlegend=True,
    template='plotly_white'
)

fig.show()
print("✅ Interactive visualization created!")

📊 Creating interactive stock price visualizations...


NameError: name 'stock_data' is not defined

In [ ]:
# Cell 4: Statistical Analysis and Correlation Heatmap
print("📊 Creating statistical analysis...")

# Create correlation matrix
correlation_data = price_data
correlation_matrix = np.corrcoef(correlation_data.T)

# Create heatmap
fig_corr = px.imshow(
    correlation_matrix,
    labels=dict(x="Stock", y="Stock", color="Correlation"),
    x=stock_symbols,
    y=stock_symbols,
    color_continuous_scale='RdBu',
    title='📊 Stock Price Correlation Matrix'
)

fig_corr.update_layout(width=600, height=500)
fig_corr.show()

# Create distribution plot
fig_dist = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Price Distribution', 'Returns Distribution', 'Volume Distribution', 'Volatility Distribution')
)

# Price distribution
for i, symbol in enumerate(stock_symbols[:2]):
    fig_dist.add_trace(
        go.Histogram(
            x=price_data[:, i],
            name=symbol,
            opacity=0.7,
            nbinsx=50
        ),
        row=1, col=1
    )

# Returns distribution
for i, symbol in enumerate(stock_symbols[:2]):
    fig_dist.add_trace(
        go.Histogram(
            x=returns_data[:, i],
            name=f'{symbol} Returns',
            opacity=0.7,
            nbinsx=50
        ),
        row=1, col=2
    )

# Volatility calculation
volatility = np.std(returns_data, axis=0)
fig_dist.add_trace(
    go.Bar(
        x=stock_symbols[:len(volatility)],
        y=volatility,
        name='Volatility',
        showlegend=False
    ),
    row=2, col=2
)

fig_dist.update_layout(
    title='📈 Statistical Analysis Dashboard',
    height=600,
    showlegend=True,
    template='plotly_white'
)

fig_dist.show()
print("✅ Statistical analysis completed!")

## 🤖 Step 2: Train Prediction Models

In [ ]:
# Cell 5: Train Transformer Model
print("🤖 Training Transformer model...")

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
args.device = device
args.model = "Transformer"

# Initialize model
transformer_model = PM_Transformer(args, stock_data)

print(f"📊 Model initialized on {device}")
print(f"🎯 Training epochs: {args.train_epochs}")
print(f"📈 Sequence length: {args.seq_len}")
print(f"🔮 Prediction length: {args.prediction_len}")

# Train the model
print("\n🚀 Starting training...")
try:
    transformer_model.train()
    print("✅ Transformer training completed!")
except Exception as e:
    print(f"⚠️ Training encountered issues: {e}")
    print("🔄 Using pre-trained model if available...")
    
    # Try to load pre-trained model
    checkpoint_dir = os.path.join("checkpoints", "transformer")
    if os.path.exists(checkpoint_dir):
        checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')]
        if checkpoint_files:
            checkpoint_path = os.path.join(checkpoint_dir, checkpoint_files[0])
            transformer_model.model.load_state_dict(torch.load(checkpoint_path, map_location=device))
            print(f"✅ Loaded pre-trained model: {checkpoint_files[0]}")

In [ ]:
# Cell 6: Train LSTM Model
print("🤖 Training LSTM model...")

# Load LSTM configuration
with open("model_config/alstm_config.json", 'r') as f:
    lstm_config = json.load(f)
args_lstm = dict_to_namespace(lstm_config)
args_lstm.device = device
args_lstm.model = "ALSTM"

# Load LSTM data
stock_data_lstm = Stock_Data(
    dataset_name=args_lstm.data_dict[args_lstm.project_name]["dataset_name"],
    full_stock_path=args_lstm.data_dict[args_lstm.project_name]["full_stock_path"],
    window_size=args_lstm.seq_len,
    root_path=args_lstm.root_path,
    prediction_len=args_lstm.prediction_len,
    scale=True
)

# Initialize LSTM model
lstm_model = PM_LSTM(args_lstm, stock_data_lstm)

print(f"📊 LSTM Model initialized on {device}")
print(f"🎯 Hidden size: {args_lstm.model_config['hidden_size']}")
print(f"📈 Num layers: {args_lstm.model_config['num_layers']}")

# Train the LSTM model
print("\n🚀 Starting LSTM training...")
try:
    lstm_model.train()
    print("✅ LSTM training completed!")
except Exception as e:
    print(f"⚠️ LSTM training encountered issues: {e}")
    print("🔄 Using pre-trained model if available...")
    
    # Try to load pre-trained model
    checkpoint_dir = os.path.join("checkpoints", "lstm")
    if os.path.exists(checkpoint_dir):
        checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')]
        if checkpoint_files:
            checkpoint_path = os.path.join(checkpoint_dir, checkpoint_files[0])
            lstm_model.model.load_state_dict(torch.load(checkpoint_path, map_location=device))
            print(f"✅ Loaded pre-trained LSTM model: {checkpoint_files[0]}")